In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from exactextract import exact_extract



In [ ]:

# =============================================================================
# Paths
# =============================================================================
main =  r"C:\Users\eunic\Dropbox\sa_fires"
int_path = Path(
    r"C:\Users\eunic\Dropbox\sa_fires"
    r"\proj_downwind\data_output\intermediate"
)

input_path = Path(
    r"C:\Users\eunic\Dropbox\sa_fires"
    r"\data\input\population_density\sedac\gpw-v4-population-density-rev11_2010_30_sec_tif"
)

output_path = fr"{main}/proj_bureaucrats_farms\data_output\intermediate"

    
grid_path = int_path / "1-grid-generation.shp"
population_raster_path = input_path / "gpw_v4_population_density_rev11_2010_30_sec.tif"

grid_population_output = (
    int_path / "small_grid_population_2010.parquet"
)

# CRS previously used in the R code for metric calculations
metric_crs = "EPSG:7755"

# Set to True only if the raster stores persons/km².
# Leave False if each pixel stores a population count.
raster_is_population_density = True


# =============================================================================
# Read grids
# =============================================================================

grids = gpd.read_file(
    grid_path,
    engine="pyogrio",
    use_arrow=True
)

# Shapefiles truncate long column names
if "unique_small_grid_id" not in grids.columns:
    if "unq_s__" in grids.columns:
        grids = grids.rename(
            columns={"unq_s__": "unique_small_grid_id"}
        )
    else:
        raise KeyError(
            "Could not find unique_small_grid_id or unq_s__."
        )

grids["unique_small_grid_id"] = pd.to_numeric(
    grids["unique_small_grid_id"],
    errors="raise"
).astype("int64")

if grids["unique_small_grid_id"].duplicated().any():
    duplicate_ids = (
        grids.loc[
            grids["unique_small_grid_id"].duplicated(False),
            "unique_small_grid_id"
        ]
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    raise ValueError(
        "The grid shapefile has duplicated grid IDs. "
        f"Examples: {duplicate_ids}"
    )

if grids.crs is None:
    raise ValueError("The grid shapefile has no CRS.")

grids = grids[
    ["unique_small_grid_id", "geometry"]
].copy()


# =============================================================================
# Raster CRS
# =============================================================================

with rasterio.open(population_raster_path) as src:
    raster_crs = src.crs
    raster_nodata = src.nodata
    raster_shape = src.shape

if raster_crs is None:
    raise ValueError("The population raster has no CRS.")

print("Grid CRS:", grids.crs)
print("Raster CRS:", raster_crs)
print("Raster shape:", raster_shape)
print("Raster nodata:", raster_nodata)


# =============================================================================
# Reproject polygons to the raster CRS
# =============================================================================

grids_raster_crs = grids.to_crs(raster_crs)


# =============================================================================
# Area-weighted population extraction
# =============================================================================

if raster_is_population_density:
    # For density rasters measured in persons/km²
    population_operation = (
        "population_2010="
        "sum(coverage_weight=area_spherical_km2)"
    )
else:
    # For population-count rasters:
    # pixel population × fractional polygon coverage
    population_operation = "population_2010=sum"


In [ ]:

grid_population = exact_extract(
    str(population_raster_path),
    grids_raster_crs,
    population_operation,
    include_cols="unique_small_grid_id",
    output="pandas"
)

grid_population["unique_small_grid_id"] = pd.to_numeric(
    grid_population["unique_small_grid_id"],
    errors="raise"
).astype("int64")

grid_population["population_2010"] = (
    pd.to_numeric(
        grid_population["population_2010"],
        errors="coerce"
    )
    .fillna(0.0)
    .astype("float64")
)


# =============================================================================
# Calculate static centroids and grid areas
# =============================================================================

grids_metric = grids.to_crs(metric_crs)

centroids = grids_metric.geometry.centroid

grid_coordinates = pd.DataFrame(
    {
        "unique_small_grid_id":
            grids_metric["unique_small_grid_id"].to_numpy(),

        "centroid_x":
            centroids.x.to_numpy(dtype="float64"),

        "centroid_y":
            centroids.y.to_numpy(dtype="float64"),

        "grid_area_km2":
            grids_metric.geometry.area.to_numpy(
                dtype="float64"
            ) / 1_000_000
    }
)

grid_population = (
    grid_population
    .merge(
        grid_coordinates,
        on="unique_small_grid_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("unique_small_grid_id")
    .reset_index(drop=True)
)


# =============================================================================
# Validation
# =============================================================================

required_numeric = [
    "population_2010",
    "centroid_x",
    "centroid_y",
    "grid_area_km2"
]

if not np.isfinite(
    grid_population[required_numeric].to_numpy()
).all():
    raise ValueError(
        "The static grid table contains non-finite values."
    )

if (grid_population["population_2010"] < 0).any():
    raise ValueError("Negative grid population detected.")

if (grid_population["grid_area_km2"] <= 0).any():
    raise ValueError("Non-positive grid area detected.")

print(grid_population.head())

print(
    "\nTotal population across grids:",
    grid_population["population_2010"].sum()
)

print(
    "Number of grids:",
    len(grid_population)
)


# =============================================================================
# Save the reusable static grid table
# =============================================================================

grid_population.to_parquet(
    grid_population_output,
    index=False,
    compression="zstd"
)

print(
    "\nSaved:",
    grid_population_output
)

In [ ]:
grid_population